In [ ]:
pip install pandas numpy scikit-learn matplotlib scipy tensorflow

#Step 1: Collect and Preprocess Stock Data

In [ ]:
# Import required libraries
import pandas as pd
import yfinance as yf

# Download stock data from Yahoo Finance
stock_data = yf.download('AAPL', start='2010-01-01', end='2022-02-26')

# Print first few rows of data
print(stock_data.head())

# Convert data to Pandas DataFrame
df = pd.DataFrame(stock_data)

# Flatten MultiIndex columns if they exist (assuming second level is the ticker)
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.droplevel(1)

# Print data types of each column to confirm structure
print(df.dtypes)
# Print the new column names to confirm the flattening
print(df.columns)


/tmp/ipython-input-139/2215461032.py:6: FutureWarning: YF.download() has changed argument auto_adjust default to True
  stock_data = yf.download('AAPL', start='2010-01-01', end='2022-02-26')
[*********************100%***********************]  1 of 1 completed

Price          Close      High       Low      Open     Volume
Ticker          AAPL      AAPL      AAPL      AAPL       AAPL
Date                                                         
2010-01-04  6.412384  6.427066  6.363544  6.395005  493729600
2010-01-05  6.423471  6.459726  6.389612  6.430063  601904800
2010-01-06  6.321294  6.448936  6.314702  6.423468  552160000
2010-01-07  6.309609  6.352157  6.263766  6.344667  477131200
2010-01-08  6.351557  6.352157  6.264066  6.301219  447610800
Price
Close     float64
High      float64
Low       float64
Open      float64
Volume      int64
dtype: object
Index(['Close', 'High', 'Low', 'Open', 'Volume'], dtype='object', name='Price')


#Step 2: Split Data into Training and Testing Sets

In [ ]:
# Split data into training and testing sets
train_size = int(0.8 * len(df))
train_data, test_data = df[:train_size], df[train_size:]

# Print sizes of training and testing sets
print(f'Train size: {len(train_data)}')
print(f'Test size: {len(test_data)}')

Train size: 2447
Test size: 612


#Step 3: Create Technical Indicators

In [ ]:
# Create moving average technical indicator
ma_50 = train_data['Close'].rolling(window=50).mean()
ma_200 = train_data['Close'].rolling(window=200).mean()

# Create relative strength index (RSI) technical indicator
delta = train_data['Close'].diff()
up_days = delta.copy()
up_days[delta <= 0] = 0
down_days = abs(delta.copy())
down_days[delta > 0] = 0
rs = up_days.ewm(com=13-1, adjust=False).mean() / down_days.ewm(com=13-1, adjust=False).mean()
rsi = 100 - (100 / (1 + rs))

# Print first few rows of technical indicators
print(ma_50.head())
print(rsi.head())

Date
2010-01-04   NaN
2010-01-05   NaN
2010-01-06   NaN
2010-01-07   NaN
2010-01-08   NaN
Name: Close, dtype: float64
Date
2010-01-04           NaN
2010-01-05    100.000000
2010-01-06     56.561111
2010-01-07     53.672772
2010-01-08     61.349114
Name: Close, dtype: float64


#Step 4: Train Machine Learning Model

In [ ]:
# Import required libraries
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

# Add technical indicators to train_data
train_data['MA_50'] = ma_50
train_data['MA_200'] = ma_200
train_data['RSI'] = rsi

# Drop rows with NaN values (which are present at the beginning due to rolling window calculations)
train_data.dropna(inplace=True)

# Create feature matrix
X = train_data[['MA_50', 'MA_200', 'RSI']]

# Create target variable
y = train_data['Close']

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train linear regression model
model = LinearRegression()
model.fit(X_train, y_train)

# Make predictions on testing set
y_pred = model.predict(X_test)

# Print mean squared error
print(mean_squared_error(y_test, y_pred))

1.1903006397959237


/tmp/ipython-input-139/3578098204.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_data['MA_50'] = ma_50
/tmp/ipython-input-139/3578098204.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_data['MA_200'] = ma_200
/tmp/ipython-input-139/3578098204.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide

#Code Examples
Example 1: Predicting Stock Prices using ARIMA

In [ ]:
# Import required libraries
import pandas as pd
from statsmodels.tsa.arima.model import ARIMA

# Download stock data from Yahoo Finance
stock_data = yf.download('AAPL', start='2010-01-01', end='2022-02-26')

# Convert data to Pandas DataFrame
df = pd.DataFrame(stock_data)

# Create ARIMA model
model = ARIMA(df['Close'], order=(5,1,0))

# Fit model
model_fit = model.fit()

# Print summary of model
print(model_fit.summary())

/tmp/ipython-input-139/3305733570.py:6: FutureWarning: YF.download() has changed argument auto_adjust default to True
  stock_data = yf.download('AAPL', start='2010-01-01', end='2022-02-26')
[*********************100%***********************]  1 of 1 completed
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dat

                               SARIMAX Results                                
Dep. Variable:                   AAPL   No. Observations:                 3059
Model:                 ARIMA(5, 1, 0)   Log Likelihood               -4607.029
Date:                Thu, 26 Feb 2026   AIC                           9226.058
Time:                        03:05:43   BIC                           9262.211
Sample:                             0   HQIC                          9239.049
                               - 3059                                         
Covariance Type:                  opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
ar.L1         -0.0499      0.008     -6.538      0.000      -0.065      -0.035
ar.L2          0.0104      0.007      1.444      0.149      -0.004       0.025
ar.L3         -0.0395      0.008     -4.751      0.0

#Example 2: Predicting Stock Prices using LSTM

In [ ]:
# Import required libraries
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from keras.models import Sequential
from keras.layers import Dense, LSTM, Dropout
import numpy as np # Import numpy for array manipulation

# Download stock data from Yahoo Finance
stock_data = yf.download('AAPL', start='2010-01-01', end='2022-02-26')

# Convert data to Pandas DataFrame
df = pd.DataFrame(stock_data)

# Flatten MultiIndex columns if they exist (assuming second level is the ticker)
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.droplevel(1)

# Select features for LSTM
features = ['Open', 'High', 'Low', 'Close']
data_for_lstm = df[features].values

# Scale data
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler.fit_transform(data_for_lstm)

# Define lookback window (number of past timesteps to consider for prediction)
lookback_window = 10 # This matches the input_shape=(10, 4) specified in the original LSTM layer

# Create sequences for LSTM
X_lstm, y_lstm = [], []
for i in range(len(scaled_data) - lookback_window):
    X_lstm.append(scaled_data[i:(i + lookback_window), :])
    y_lstm.append(scaled_data[i + lookback_window, 3]) # Predict 'Close' price, which is the 4th column (index 3)

X_lstm, y_lstm = np.array(X_lstm), np.array(y_lstm)

# Create LSTM model
model = Sequential()
# Input shape is (timesteps, features)
model.add(LSTM(50, activation='relu', input_shape=(lookback_window, len(features))))
model.add(Dropout(0.2))
model.add(Dense(1)) # Output a single value (Close price)

# Compile model
model.compile(loss='mean_squared_error', optimizer='adam')

# Fit model
# X_lstm is already 3D and y_lstm is 1D, matching the LSTM input expectations
model.fit(X_lstm, y_lstm, epochs=10, batch_size=32, verbose=1) # Reduced epochs for faster execution

# Make predictions
predictions_scaled = model.predict(X_lstm)

# Inverse transform predictions to original scale
# Create a dummy array to inverse transform the single predicted 'Close' value back.
# The scaler was fitted on 4 features, so we need a 4-column array.
dummy_predictions = np.zeros(shape=(len(predictions_scaled), len(features)))
dummy_predictions[:, 3] = predictions_scaled.flatten() # Place predicted 'Close' values in the 'Close' column
predictions = scaler.inverse_transform(dummy_predictions)[:, 3] # Inverse transform and extract the 'Close' column

# Inverse transform actual values for comparison
dummy_actual = np.zeros(shape=(len(y_lstm), len(features)))
dummy_actual[:, 3] = y_lstm.flatten()
actual_values = scaler.inverse_transform(dummy_actual)[:, 3]

# Optional: Print first few predictions and actuals for verification
print("First 10 predictions:", predictions[:10])
print("First 10 actual values:", actual_values[:10])

/tmp/ipython-input-139/1998509186.py:9: FutureWarning: YF.download() has changed argument auto_adjust default to True
  stock_data = yf.download('AAPL', start='2010-01-01', end='2022-02-26')
[*********************100%***********************]  1 of 1 completed

Epoch 1/10



/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


96/96 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 0.0328
Epoch 2/10
96/96 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - loss: 0.0021
Epoch 3/10
96/96 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 0.0016
Epoch 4/10
96/96 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.0017
Epoch 5/10
96/96 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0012
Epoch 6/10
96/96 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 0.0014
Epoch 7/10
96/96 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.0011
Epoch 8/10
96/96 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0012
Epoch 9/10
96/96 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 9.6884e-04
Epoch 10/10
96/96 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 9.9303e-04
96/96 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
First 10 predictions: [7.88517084 7.88629443 7.8841519  7.87973844 7.85902138 7.83908891
 7.83750946 7.83337915 7.81758598 7.78799449]
First 10 actual values: [6.44324636 6.34406662 6.2344017  5.92518473 6.08458805 6.17058277
 6.22870922 5.97132778 5.75469589 5.83469534]
